# TMA2 - Sitting Posture Classification (DenseNet121)

## 1. Chuẩn bị Google Colab


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123



## Dataset chuẩn 5 nhãn

Upload `posture_dataset_standard_5class.zip`. Cả bốn notebook dùng cùng train/val/test; không tải split gốc từ Roboflow.

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

ZIP_PATH = Path("/content/posture_dataset_standard_5class.zip")
DATA_DIR = Path("/content/posture_dataset_standard_5class")
CLASS_NAMES = ["leaning_backward", "leaning_forward", "leaning_left", "leaning_right", "upright"]

if not DATA_DIR.is_dir():
    if not ZIP_PATH.is_file():
        print("Chọn posture_dataset_standard_5class.zip để upload lên Colab")
        uploaded = files.upload()
        assert ZIP_PATH.name in uploaded, f"Cần upload đúng file {ZIP_PATH.name}"
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall("/content")

TRAIN_DIR = str(DATA_DIR / "train")
VAL_DIR = str(DATA_DIR / "val")
TEST_DIR = str(DATA_DIR / "test")
for split_dir in (TRAIN_DIR, VAL_DIR, TEST_DIR):
    assert Path(split_dir).is_dir(), f"Thiếu split: {split_dir}"
    assert sorted(p.name for p in Path(split_dir).iterdir() if p.is_dir()) == CLASS_NAMES, f"Sai 5 nhãn trong {split_dir}"
print("Dataset chuẩn:", DATA_DIR)
print("Thứ tự nhãn:", CLASS_NAMES)


In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_names=CLASS_NAMES, label_mode="int", shuffle=True, seed=SEED
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_names=CLASS_NAMES, label_mode="int", shuffle=False
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_names=CLASS_NAMES, label_mode="int", shuffle=False
)
class_names = list(CLASS_NAMES)
assert train_ds.class_names == val_ds.class_names == test_ds.class_names == class_names
assert len(class_names) == 5
print("Classes:", class_names)


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize = (9, 9))
for images, labels in train_ds.take(1):
    n = min(9, len(images))
    for i in range(n):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)


In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10),
    ],
    name="data_augmentation"
)


## 2. Xây dựng DenseNet121 — phiên bản ổn định cho dataset nhỏ

Chiến lược:
- Phase 1: đóng băng toàn bộ DenseNet121, chỉ train classification head.
- Đánh giá Phase 1 riêng trên test set.
- Phase 2: load lại checkpoint tốt nhất Phase 1, chỉ mở 30 layer cuối.
- Fine-tune nhẹ với learning rate `1e-6`.
- Lưu checkpoint Phase 1 và Phase 2 riêng.
- Cuối cùng tự động chọn model có test accuracy tốt hơn.

Lưu ý: Không dùng `RandomFlip("horizontal")` vì sẽ làm đảo `leaning_left` và `leaning_right`.


In [ ]:
# Xây dựng DenseNet121

base_model = tf.keras.applications.DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = keras.Input(
    shape=(224, 224, 3),
    name="input_image"
)

x = data_augmentation(inputs)

# DenseNet121 preprocessing nằm BÊN TRONG model.
# Frontend TensorFlow.js sau này chỉ cần đưa RGB 0-255 vào.
x = tf.keras.applications.densenet.preprocess_input(x)

x = base_model(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    len(class_names),
    activation="softmax",
    name="posture"
)(x)

model = keras.Model(
    inputs,
    outputs,
    name="PostureGuard_DenseNet121"
)

model.summary()


### 3. Phase 1 — Train classification head


In [ ]:
PHASE1_PATH = "/content/densenet121_phase1_best.keras"

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_phase1 = [
    keras.callbacks.ModelCheckpoint(
        PHASE1_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=callbacks_phase1
)


In [ ]:
# Đánh giá Phase 1 ngay trước khi fine-tune

phase1_model = tf.keras.models.load_model(
    PHASE1_PATH,
    compile=False
)

phase1_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

loss1, acc1 = phase1_model.evaluate(
    test_ds,
    verbose=1
)

print()
print("Phase 1 test loss    :", loss1)
print("Phase 1 test accuracy:", acc1)
print("Best val accuracy Phase 1:",
      max(history_phase1.history["val_accuracy"]))


### 4. Phase 2 — Fine-tune nhẹ

Chỉ mở 30 layer cuối của DenseNet121 và dùng learning rate rất nhỏ `1e-6`.
Model Phase 2 luôn bắt đầu từ checkpoint Phase 1 tốt nhất.


In [ ]:
# Load lại checkpoint Phase 1 tốt nhất trước khi fine-tune

model = tf.keras.models.load_model(
    PHASE1_PATH,
    compile=False
)

# Tìm backbone DenseNet121 bên trong model
base_model = None

for layer in model.layers:
    if isinstance(layer, tf.keras.Model) and "densenet" in layer.name.lower():
        base_model = layer
        break

assert base_model is not None, "Không tìm thấy DenseNet121 backbone."

print("Backbone:", base_model.name)

base_model.trainable = True

# Freeze tất cả trừ 30 layer cuối
for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers[-30:]:
    layer.trainable = True

print(
    "Trainable DenseNet backbone layers:",
    sum(int(layer.trainable) for layer in base_model.layers)
)


In [ ]:
PHASE2_PATH = "/content/densenet121_phase2_best.keras"

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-6
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_phase2 = [
    keras.callbacks.ModelCheckpoint(
        PHASE2_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True,
        verbose=1
    )
]

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks_phase2
)


### 5. So sánh Phase 1 và Phase 2, tự động chọn model tốt hơn


In [ ]:
phase1_model = tf.keras.models.load_model(
    PHASE1_PATH,
    compile=False
)

phase2_model = tf.keras.models.load_model(
    PHASE2_PATH,
    compile=False
)

for m in [phase1_model, phase2_model]:
    m.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

loss1, acc1 = phase1_model.evaluate(
    test_ds,
    verbose=0
)

loss2, acc2 = phase2_model.evaluate(
    test_ds,
    verbose=0
)

print("Phase 1 test accuracy:", acc1)
print("Phase 2 test accuracy:", acc2)

print("Best val accuracy Phase 1:",
      max(history_phase1.history["val_accuracy"]))
print("Best val accuracy Phase 2:",
      max(history_phase2.history["val_accuracy"]))

if acc2 > acc1:
    best_model = phase2_model
    best_phase = "Phase 2"
else:
    best_model = phase1_model
    best_phase = "Phase 1"

print()
print("Selected:", best_phase)
print("Selected test accuracy:", max(acc1, acc2))


### 6. Classification report + confusion matrix + phân bố dự đoán


In [ ]:
import numpy as np
from collections import Counter
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = best_model.predict(
        images,
        verbose=0
    )

    pred_labels = np.argmax(
        preds,
        axis=1
    )

    y_true.extend(labels.numpy())
    y_pred.extend(pred_labels)

print("Classes:")
for i, name in enumerate(class_names):
    print(i, "->", name)

print()
print("True distribution:")
print(Counter(y_true))

print()
print("Prediction distribution:")
print(Counter(y_pred))

print()
print("Classification report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_true,
        y_pred
    )
)


In [ ]:
# Vẽ confusion matrix

import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm)

ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))

ax.set_xticklabels(
    class_names,
    rotation=45,
    ha="right"
)
ax.set_yticklabels(class_names)

ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(
    f"DenseNet121 Confusion Matrix - {best_phase}"
)

for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar(im)
plt.tight_layout()
plt.show()


### 7. Lưu model tốt nhất


In [ ]:
FINAL_KERAS_PATH = "/content/densenet121_posture_v1.keras"

best_model.save(
    FINAL_KERAS_PATH
)

print("Selected phase:", best_phase)
print("Saved Keras model:", FINAL_KERAS_PATH)


### 8. Export SavedModel để convert sang TensorFlow.js


In [ ]:
SAVED_MODEL_PATH = "/content/densenet121_saved_model"

best_model.export(
    SAVED_MODEL_PATH
)

print("SavedModel:", SAVED_MODEL_PATH)


In [ ]:
# ZIP + download SavedModel

import os
import shutil
from google.colab import files

ZIP_BASE = "/content/densenet121_saved_model"
ZIP_PATH = ZIP_BASE + ".zip"

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

import json
with open(os.path.join(SAVED_MODEL_PATH, "class_names.json"), "w") as stream:
    json.dump(class_names, stream)

shutil.make_archive(
    ZIP_BASE,
    "zip",
    SAVED_MODEL_PATH
)

print("ZIP created:", ZIP_PATH)

files.download(ZIP_PATH)


### 9. Convert local trên macOS

Sau khi tải `densenet121_saved_model.zip`, giải nén và chạy trong Python 3.11 environment:

```bash
tensorflowjs_converter \
  --input_format=tf_saved_model \
  --output_format=tfjs_graph_model \
  --signature_name=serving_default \
  --saved_model_tags=serve \
  densenet121_saved_model \
  densenet121_tfjs_model
```

Kết quả cần có `model.json` và các file `group*.bin`.
Upload vào PostureGuard Admin với model key `densenet121`.
